In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('housing_data.csv',skipinitialspace=True)

In [ ]:
df.info()
df.duplicated().sum()
df.isnull().sum()
df.describe(include='all')

In [ ]:
Q1 = df['total_bedrooms'].quantile(0.25)
Q3 = df['total_bedrooms'].quantile(0.75)
IQR = Q3 - Q1

# Tính toán 2 đường ranh giới
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 1. Lọc ra danh sách các row chứa Outlier để kiểm tra
outliers = df[(df['total_bedrooms'] < lower_bound) | (df['total_bedrooms'] > upper_bound)]
print(f"Số lượng outlier tìm thấy: {len(outliers)}")

# 2. Hoặc Lọc bỏ Outlier (chỉ giữ lại các dòng nằm trong khoảng an toàn)
df_cleaned = df[(df['total_bedrooms'] >= lower_bound) & (df['total_bedrooms'] <= upper_bound)]

In [ ]:
# Clean data (fill missing total_bedrooms with median)
df_head = df.head(10).copy()
median_bedrooms = df_head['total_bedrooms'].median()
df_head['total_bedrooms'] = df_head['total_bedrooms'].fillna(median_bedrooms)

print("Data after filling missing values (first 10 rows):")
print(df_head.isnull().sum())

In [ ]:
# Scatter plot to find outliers (median_income vs median_house_value)
plt.figure(figsize=(10, 6))
sns.scatterplot(x='median_income', y='median_house_value', data=df_head)
plt.title('Scatter plot: Median Income vs Median House Value (First 10 rows)')
plt.show()

In [ ]:
# --- Yêu cầu mới: Đọc 10 dòng đầu nhưng áp dụng cho toàn bộ dataset ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Giả lập việc "chỉ đọc 10 dòng đầu" (mô phỏng logic)
# Nhưng xử lý trên TẤT CẢ dữ liệu (bỏ .head(10))

# 1. Clean data (fill missing total_bedrooms with median for ENTIRE dataset)
median_bedrooms_all = df['total_bedrooms'].median()
df['total_bedrooms'] = df['total_bedrooms'].fillna(median_bedrooms_all)

print("Data after filling missing values (ALL rows):")
print(df.isnull().sum())

In [ ]:
# 2. Scatter plot to find outliers (median_income vs median_house_value for ENTIRE dataset)
plt.figure(figsize=(10, 6))
sns.scatterplot(x='median_income', y='median_house_value', data=df, alpha=0.5)
plt.title('Scatter plot: Median Income vs Median House Value (ALL rows)')
plt.show()

In [ ]:
# 3. Find correlation between features (ENTIRE dataset)
correlation_matrix_all = df.select_dtypes(include=[np.number]).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix_all, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix (ALL rows)')
plt.show()

In [ ]:
# 1. Vẽ scatter plot latitude vs longitude với s=population/100, c=median_house_value, cmap='jet'
plt.figure(figsize=(10, 7))
scatter = plt.scatter(x=df['longitude'], y=df['latitude'], 
            alpha=0.4, 
            s=df['population']/100, label='population', 
            c=df['median_house_value'], cmap=plt.get_cmap('jet'))
plt.title("California Housing Prices: Location, Population, and Value")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.colorbar(scatter, label='Median House Value')
plt.legend()
plt.show()

In [ ]:
# 2. Tính correlation matrix với median_house_value
corr_matrix = df.corr(numeric_only=True)
corr_with_target = corr_matrix['median_house_value'].sort_values(ascending=False)
print("Mức độ tương quan của các feature với median_house_value:")
print(corr_with_target)

In [ ]:
# 3. Tạo 3 features mới
df['rooms_per_house'] = df['total_rooms'] / df['households']
df['bedrooms_ratio'] = df['total_bedrooms'] / df['total_rooms']
df['people_per_house'] = df['population'] / df['households']

print("Dữ liệu sau khi thêm 3 features mới:")
df[['rooms_per_house', 'bedrooms_ratio', 'people_per_house']].head()

In [ ]:
# 4. Tính lại correlation - xem các feature mới có cải thiện không
new_corr_matrix = df.corr(numeric_only=True)
new_corr_with_target = new_corr_matrix['median_house_value'].sort_values(ascending=False)

print("Correlation với median_house_value sau khi thêm features mới:")
print(new_corr_with_target)

In [ ]:
# 5. Dùng scatter_matrix() để vẽ top 4 features tương quan cao nhất
from pandas.plotting import scatter_matrix

# Dựa vào kết quả ở trên, top 4 attributes tương quan mạnh (hoặc có ý nghĩa) nhất (bỏ target đi thì sẽ là median_income, rooms_per_house, bedrooms_ratio, housing_median_age v.v.)
# Lấy tên của 4 features có độ lớn tương quan (absolute value) cao nhất với biến mục tiêu (bao gồm cả target)
# (Loại target column, lấy top 3 features có ảnh hưởng nhất + target = top 4)
top_4_attributes = new_corr_with_target.abs().sort_values(ascending=False).head(4).index.tolist()

print(f"Top 4 features để vẽ scatter_matrix: {top_4_attributes}")

# Vẽ scatter_matrix
scatter_matrix(df[top_4_attributes], figsize=(12, 8))
plt.show()

In [ ]:
# ===== BÀI TẬP 4: XÂY DỰNG PREPROCESSING PIPELINE =====

# Bước 1: Tạo stratified train-test split
from sklearn.model_selection import train_test_split

# Tạo income category để stratify
df['income_cat'] = pd.cut(df['median_income'],
                          bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                          labels=[1, 2, 3, 4, 5])

# Stratified split
from sklearn.model_selection import StratifiedShuffleSplit
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(df, df['income_cat']):
    strat_train_set = df.loc[train_index]
    strat_test_set = df.loc[test_index]

# Xóa income_cat column
for set_ in (strat_train_set, strat_test_set):
    set_.drop('income_cat', axis=1, inplace=True)

print(f"Training set size: {len(strat_train_set)}")
print(f"Test set size: {len(strat_test_set)}")

In [ ]:
# Bước 2: Tách X_train và y_train
X_train = strat_train_set.drop('median_house_value', axis=1)
y_train = strat_train_set['median_house_value'].copy()

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"\nX_train columns:\n{X_train.columns.tolist()}")

In [ ]:
# Bước 3: Tạo num_pipeline (SimpleImputer → StandardScaler)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('std_scaler', StandardScaler())
])

print("num_pipeline created successfully!")
print(num_pipeline)

In [ ]:
# Bước 4: Tạo cat_pipeline (SimpleImputer(strategy='most_frequent') → OneHotEncoder)
from sklearn.preprocessing import OneHotEncoder

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder())
])

print("cat_pipeline created successfully!")
print(cat_pipeline)

In [ ]:
# Bước 5: Kết hợp bằng ColumnTransformer
from sklearn.compose import ColumnTransformer

# Xác định các cột numerical và categorical
num_attribs = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_attribs = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns: {num_attribs}")
print(f"Categorical columns: {cat_attribs}")

# Tạo ColumnTransformer
preprocessing = ColumnTransformer([
    ('num', num_pipeline, num_attribs),
    ('cat', cat_pipeline, cat_attribs)
])

print("\nColumnTransformer created successfully!")
print(preprocessing)

In [ ]:
# Bước 6: Gọi preprocessing.fit_transform(X_train) và kiểm tra shape output
X_train_prepared = preprocessing.fit_transform(X_train)

print(f"Original X_train shape: {X_train.shape}")
print(f"Transformed X_train_prepared shape: {X_train_prepared.shape}")
print(f"\nX_train_prepared type: {type(X_train_prepared)}")
print(f"\nFirst 5 rows of transformed data:")
print(X_train_prepared[:5])